# Notebook 02 - RAG Chain & Conversational Q&A

**Project:** IncidentIQ - AI-powered Incident Intelligence

**Goal:** Build a conversational Q&A chain on top of the Pinecone knowledge base.

## What this notebook covers
1. Install and import all required packages
2. Configuration and Pinecone initialization
3. System prompt with language and citation rules
4. Retriever with query rewriting and multi-query
5. Conversation memory
6. RAG chain with ask() function
7. Test English and Dutch Q&A
8. Quality tests

---

## 1. Install required packages

In [ ]:
!pip install langchain langchain-openai langchain-community langchain-pinecone pinecone python-dotenv -q
print('Packages installed.')

## 2. Import libraries and load environment variables

In [ ]:
import os
import re
import json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pinecone import Pinecone

load_dotenv()
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY not found.'
assert os.getenv('PINECONE_API_KEY'), 'PINECONE_API_KEY not found.'
print('Environment variables loaded successfully.')

## 3. Configuration

Must match the values used in Notebook 01.

In [ ]:
INDEX_NAME      = 'incidentiq'
EMBEDDING_MODEL = 'text-embedding-3-small'
LLM_MODEL       = 'gpt-4o-mini'
RETRIEVER_K     = 8

llm             = ChatOpenAI(model=LLM_MODEL, temperature=0)
embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)
pc              = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
vectorstore     = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    pinecone_api_key=os.getenv('PINECONE_API_KEY'),
)

print('Configuration set.')
print(f'   LLM model:       {LLM_MODEL}')
print(f'   Pinecone index:  {INDEX_NAME}')

## 4. System prompt

The system prompt controls language detection, citation style and answer format.
Key rules:
- Always respond in the same language as the user question
- Use bullet points — never paragraphs
- Cite timestamps as source references
- Never make up information not in the context

In [ ]:
SYSTEM_PROMPT = """You are IncidentIQ, an AI assistant specialized in incident training and knowledge extraction.
You help emergency services professionals extract knowledge from incident training videos.
You work for any organization: fire services, police, EMS, civil protection.

ANSWER RULES:
- Answer strictly based on the retrieved context below.
- Never make up information not present in the context.
- If the answer is not in the context say: I could not find this information in the video.

LANGUAGE RULE:
- Always respond in the same language as the user question.
- Dutch question -> Dutch answer with natural Belgian incident training language.
- English question -> English answer.
- French question -> French answer.

FORMAT RULE:
- Always use bullet points - never write paragraphs.
- Maximum 15 words per bullet.
- Keep answers concise and factual.

CITATION RULE:
- Do NOT include timestamps in your answer text.
- Timestamps are added automatically below your answer.

Context:
{context}

Conversation history:
{chat_history}
"""

print('System prompt defined.')
print(f'   Length: {len(SYSTEM_PROMPT)} characters')

## 5. Retriever with query rewriting and multi-query

We combine two techniques for better retrieval quality:

**Query rewriting:** The LLM rewrites the user query to be more specific
before searching. Vague queries become precise retrieval queries.

**Multi-query:** The LLM generates 3 variations of the query and retrieves
chunks for each. Results are deduplicated. More angles = better coverage.

Why this matters:
ChromaDB and Pinecone use cosine similarity on vector embeddings.
Better queries = better vectors = more relevant chunks retrieved.
This directly improves RAGAs context recall scores.

In [ ]:
def retrieve_context(query: str) -> tuple:
    """
    Retrieve relevant chunks using query rewriting and multi-query.
    Returns (clean_context_for_llm, source_timestamps).

    Steps:
    1. Translate query to English for better retrieval
    2. Rewrite query to be more specific
    3. Generate 3 query variations
    4. Retrieve chunks for each variation
    5. Deduplicate and return combined context
    """
    # Step 1 - Translate to English
    # Pinecone contains English chunks - English queries always return better results
    english_query = llm.invoke(
        f'Translate to English, return only the translation: {query}'
    ).content.strip()

    # Step 2 - Rewrite for better retrieval
    rewritten = llm.invoke(
        f'Rewrite this search query to be more specific for searching '
        f'an incident training video transcript. '
        f'Return only the rewritten query, maximum 20 words.\n\n'
        f'Query: {english_query}\nRewritten:'
    ).content.strip()

    # Step 3 - Generate multi-query variations
    try:
        response = llm.invoke(
            f'Generate 3 different versions of this question for document retrieval. '
            f'Each approaches the topic differently. '
            f'Return only a JSON list of 3 strings.\n\n'
            f'Question: {rewritten}\nJSON:'
        ).content.strip()
        raw = re.sub(r'```json|```', '', response).strip()
        queries = json.loads(raw)
    except Exception:
        queries = [rewritten]
    queries.append(rewritten)

    # Step 4 - Retrieve for each query and deduplicate
    all_docs = {}
    for q in queries:
        results = vectorstore.similarity_search(q, k=4)
        for doc in results:
            key = doc.page_content[:100]
            if key not in all_docs:
                all_docs[key] = doc

    combined = list(all_docs.values())

    if not combined:
        return 'No relevant information found. Please load a YouTube video first.', []

    # Step 5 - Extract timestamps and clean context
    all_ts = re.findall(r'\[\d{2}:\d{2}\]', ' '.join([d.page_content for d in combined]))
    seen, unique_ts = set(), []
    for t in all_ts:
        if t not in seen:
            seen.add(t)
            unique_ts.append(t)

    clean_chunks = [
        re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', d.page_content)
        for d in combined
    ]

    return '\n\n'.join(clean_chunks), unique_ts[:5]


# Test the retriever
print('Testing retriever with query rewriting and multi-query...')
context, timestamps = retrieve_context('What went wrong during the incident?')
print(f'Retrieved context length: {len(context)} characters')
print(f'Timestamps found: {timestamps[:3]}')
print(f'\nContext preview (first 300 chars):')
print('-' * 60)
print(context[:300])

## 6. Conversation memory

The agent remembers previous questions and answers within a session.
This allows natural follow-up questions without repeating context.
We keep the last 10 exchanges to avoid exceeding the LLM context window.

In [ ]:
chat_history = []

def format_chat_history(history: list) -> str:
    'Format chat history as readable string for the system prompt.'
    if not history:
        return 'No previous conversation.'
    lines = []
    for msg in history:
        if isinstance(msg, HumanMessage):
            lines.append(f'User: {msg.content}')
        elif isinstance(msg, AIMessage):
            lines.append(f'Assistant: {msg.content}')
    return '\n'.join(lines)

def add_to_history(history: list, question: str, answer: str) -> list:
    'Add exchange to history. Keep last 10 exchanges to avoid context overflow.'
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=answer))
    return history[-20:]

print('Conversation memory initialized.')

## 7. Build the ask() function

The ask() function is the single entry point for all Q&A interactions.
It retrieves context, generates the answer and appends source citations.

In [ ]:
def ask(question: str, verbose: bool = False) -> str:
    """
    Ask a question about the loaded video.
    Uses query rewriting and multi-query for better retrieval.
    Returns answer with timestamp citations at the bottom.
    """
    global chat_history

    # Step 1 - Retrieve context with query rewriting and multi-query
    context, timestamps = retrieve_context(question)

    if verbose:
        print(f'   Context length: {len(context)} chars')
        print(f'   Timestamps: {timestamps}')

    # Step 2 - Format source citations
    source_lines = [
        f'  Source/Timestamp {i}: {ts.strip("[]")}'  
        for i, ts in enumerate(timestamps[:3], 1)
    ]
    sources_block = '\n'.join(source_lines)

    # Step 3 - Generate answer
    prompt = ChatPromptTemplate.from_messages([
        ('system', SYSTEM_PROMPT),
        ('human', '{question}'),
    ])
    chain  = prompt | llm | StrOutputParser()
    answer = chain.invoke({
        'context':      context,
        'chat_history': format_chat_history(chat_history),
        'question':     question,
    })

    # Step 4 - Append source citations
    if sources_block:
        answer = f'{answer}\n\nSources:\n{sources_block}'

    # Step 5 - Save to memory
    chat_history = add_to_history(chat_history, question, answer)

    return answer

print('ask() function ready.')

## 8. Test English Q&A

Test the full chain with English questions.
The agent answers based on the Pinecone knowledge base.

In [ ]:
chat_history = []

print('=' * 60)
print('TEST - English Q&A')
print('=' * 60)

q1 = 'What kind of fire incident happened in Brussels?'
print(f'\nQ: {q1}')
print(f'A: {ask(q1)}')

q2 = 'What mistakes were made during the incident?'
print(f'\nQ: {q2}')
print(f'A: {ask(q2)}')

q3 = 'Can you give more detail about the first mistake you mentioned?'
print(f'\nQ: {q3}')
print(f'A: {ask(q3)}')

print(f'\nEnglish test complete - {len(chat_history)//2} exchanges in memory.')

## 9. Test Dutch Q&A

The same chain handles Dutch automatically.
Query rewriting translates Dutch to English for retrieval.
The LLM detects Dutch and responds in Dutch.

In [ ]:
chat_history = []

print('=' * 60)
print('TEST - Nederlandse Q&A')
print('=' * 60)

q1 = 'Wat voor brand was dit in Brussel?'
print(f'\nVraag: {q1}')
print(f'Antwoord: {ask(q1)}')

q2 = 'Welke fouten werden gemaakt tijdens het incident?'
print(f'\nVraag: {q2}')
print(f'Antwoord: {ask(q2)}')

q3 = 'Geef meer detail over de eerste fout die je noemde'
print(f'\nVraag: {q3}')
print(f'Antwoord: {ask(q3)}')

print(f'\nNederlandse test compleet - {len(chat_history)//2} vragen beantwoord.')

## 10. RAG chain quality tests

In [ ]:
print('=' * 60)
print('RAG CHAIN QUALITY TESTS')
print('=' * 60)

tests_passed = 0
tests_failed = 0

def check(name, condition, detail=''):
    global tests_passed, tests_failed
    if condition:
        tests_passed += 1
        print(f'  PASS - {name}')
    else:
        tests_failed += 1
        print(f'  FAIL - {name}')
    if detail:
        print(f'         {detail}')

chat_history = []

# Test 1 - English answer for English question
r = ask('What is this video about?')
check('English question returns English answer',
    len(r) > 50 and any(w in r.lower() for w in ['fire', 'incident', 'brussels', 'building']),
    f'Length: {len(r)} chars')

# Test 2 - Dutch answer for Dutch question
chat_history = []
r = ask('Waarover gaat deze video?')
check('Dutch question returns Dutch answer',
    len(r) > 50 and any(w in r.lower() for w in ['brand', 'incident', 'brussel', 'gebouw', 'de', 'het']),
    f'Length: {len(r)} chars')

# Test 3 - Answer is meaningful length
check('Answer contains enough content',
    len(r) > 100,
    f'Length: {len(r)} chars')

# Test 4 - Sources present in answer
chat_history = []
r = ask('What mistakes were made?')
check('Answer contains source citations',
    'Source' in r or 'Timestamp' in r,
    f'Preview: {r[:80]}...')

# Test 5 - Memory works for follow-up
chat_history = []
ask('What problems occurred during the incident?')
r = ask('Can you elaborate on the first problem?')
check('Memory works for follow-up questions',
    len(r) > 50,
    f'Length: {len(r)} chars')

# Test 6 - Multi-query improves retrieval
context1, _ = retrieve_context('mistakes')
context2, _ = retrieve_context('What specific errors and failures occurred?')
check('Specific query retrieves more context than vague query',
    len(context2) >= len(context1),
    f'Vague: {len(context1)} chars | Specific: {len(context2)} chars')

print('\n' + '=' * 60)
print(f'RESULTS: {tests_passed} passed | {tests_failed} failed')
if tests_failed == 0:
    print('All tests passed - RAG chain ready for Notebook 03!')
else:
    print('Fix failing tests before moving to Notebook 03.')
print('=' * 60)

---

## What we built in this notebook

| Component | What | Why |
|-----------|------|-----|
| Pinecone vectorstore | Cloud retrieval | No local storage needed |
| Query rewriting | Improves specificity | Better retrieval quality |
| Multi-query | 3 query variations | More relevant chunks found |
| System prompt | Language + format rules | Consistent behavior |
| Conversation memory | Last 10 exchanges | Natural follow-up questions |
| Source citations | Timestamps at bottom | Traceable answers |

## Next: Notebook 03 - Agent Tools